In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [2]:
rs = 100 # Globalizando random state

In [3]:
dadosNacional = pd.read_parquet("../outputs/dadosFinal.parquet")
dadosNorte = pd.read_parquet("../outputs/dadosNorte.parquet")
dadosNordeste = pd.read_parquet("../outputs/dadosNordeste.parquet")
dadosCentroOeste = pd.read_parquet("../outputs/dadosCentroOeste.parquet")
dadosSudeste = pd.read_parquet("../outputs/dadosSudeste.parquet")
dadosSul = pd.read_parquet("../outputs/dadosSul.parquet")

In [4]:
bancos = {
    "Nacional": dadosNacional,
    "Norte": dadosNorte,
    "Nordeste": dadosNordeste,
    "Centro-Oeste": dadosCentroOeste,
    "Sudeste": dadosSudeste,
    "Sul": dadosSul
}

In [5]:
dadosVarBin = {}

for regiao, dados in bancos.items():

    dados["atraso60Binario"] = dados["atraso60"].map({
        "Pontual": 0,
        "Atraso": 1
    })

    y = dados["atraso60Binario"]

    X = dados.drop(columns=[
        "atraso60",
        "atraso60Binario"
    ])

    dadosVarBin[regiao] = {
        "dados": dados,
        "X": X,
        "y": y
    }

In [6]:
for regiao, dados in dadosVarBin.items():
    X = dados["X"]
    
    categoricas = X.select_dtypes(include=['object', 'category']).columns
    numericas = X.select_dtypes(exclude=['object', 'category']).columns
    
    dados["categoricas"] = categoricas
    dados["numericas"] = numericas

In [8]:
from boruta import BorutaPy
from sklearn.preprocessing import OrdinalEncoder
from IPython.display import display
 
nAmostraBoruta = 50000 # Linhas do treino usadas em cada Boruta (amostra estratificada)
maxIterBoruta = 100
 
for regiao, dados in dadosVarBin.items():
    X = dados["X"]
    y = dados["y"]
    categoricas = dados["categoricas"]
    numericas = dados["numericas"]
 
    X_trainB, _, y_trainB, _ = train_test_split( # Mesma divisão da célula do pipeline: o Boruta nunca vê o teste
        X, y, test_size=0.3, random_state=rs, stratify=y
    )
    if len(X_trainB) > nAmostraBoruta:
        _, X_trainB, _, y_trainB = train_test_split(
            X_trainB, y_trainB, test_size=nAmostraBoruta, random_state=rs, stratify=y_trainB
        )
 
    preprocessadorBoruta = ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1), categoricas), # Uma coluna por variável (sem OneHot) e NaN vira -1
        ('num', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True), numericas)
    ])
    X_boruta = preprocessadorBoruta.fit_transform(X_trainB)
 
    arvoreBoruta = RandomForestClassifier(n_jobs=-1, max_depth=7, random_state=rs) # Árvores rasas, como o BorutaPy recomenda
    boruta = BorutaPy(
        arvoreBoruta,
        n_estimators='auto',
        perc=90, # Abaixo de 100 é menos rígido: só rejeita o que está claramente abaixo do ruído
        max_iter=maxIterBoruta,
        random_state=rs,
        verbose=0
    )
    print(f"{regiao} — Boruta rodando com {X_boruta.shape[0]} linhas e {X_boruta.shape[1]} colunas")
    boruta.fit(X_boruta, y_trainB.astype(int).to_numpy())
 
    resultadoBoruta = pd.DataFrame({
        "variavel": (
            pd.Series(preprocessadorBoruta.get_feature_names_out())
            .str.replace(r"^(cat|num)__", "", regex=True)
            .str.replace(r"^missingindicator_", "", regex=True) # O indicador de NaN conta para a variável de origem
        ),
        "status": np.where(boruta.support_, 1, np.where(boruta.support_weak_, 2, 3)),
        "ranking": boruta.ranking_
    }).groupby("variavel", as_index=False).min() # Fica com o melhor resultado entre a variável e o indicador dela
    resultadoBoruta["decisao"] = resultadoBoruta["status"].map({1: "Confirmada", 2: "Tentativa", 3: "Rejeitada"})
    resultadoBoruta = resultadoBoruta.drop(columns="status").sort_values(["ranking", "variavel"]).reset_index(drop=True)
 
    dados["boruta"] = boruta
    dados["resultadoBoruta"] = resultadoBoruta
    contagem = resultadoBoruta["decisao"].value_counts()
    print(f"{regiao} — Confirmadas: {contagem.get('Confirmada', 0)} | Tentativas: {contagem.get('Tentativa', 0)} | Rejeitadas: {contagem.get('Rejeitada', 0)}")
 
comparativoBoruta = pd.DataFrame({ # Decisão de cada variável em cada região
    regiao: dados["resultadoBoruta"].set_index("variavel")["decisao"]
    for regiao, dados in dadosVarBin.items()
})
comparativoBoruta["nRejeitada"] = (comparativoBoruta == "Rejeitada").sum(axis=1)
with pd.option_context("display.max_rows", None):
    display(comparativoBoruta.sort_values("nRejeitada", ascending=False))


Nacional — Boruta rodando com 50000 linhas e 35 colunas
Nacional — Confirmadas: 28 | Tentativas: 0 | Rejeitadas: 5
Norte — Boruta rodando com 50000 linhas e 34 colunas
Norte — Confirmadas: 30 | Tentativas: 0 | Rejeitadas: 3
Nordeste — Boruta rodando com 50000 linhas e 34 colunas
Nordeste — Confirmadas: 24 | Tentativas: 0 | Rejeitadas: 9
Centro-Oeste — Boruta rodando com 50000 linhas e 34 colunas
Centro-Oeste — Confirmadas: 31 | Tentativas: 0 | Rejeitadas: 2
Sudeste — Boruta rodando com 50000 linhas e 33 colunas
Sudeste — Confirmadas: 27 | Tentativas: 1 | Rejeitadas: 5
Sul — Boruta rodando com 50000 linhas e 35 colunas
Sul — Confirmadas: 30 | Tentativas: 0 | Rejeitadas: 3


,Nacional,Norte,Nordeste,Centro-Oeste,Sudeste,Sul,nRejeitada
variavel,,,,,,,
RegiaoResidencia,Confirmada,Rejeitada,Rejeitada,Rejeitada,Rejeitada,Rejeitada,5
comportamentoTumor,Confirmada,Rejeitada,Rejeitada,Rejeitada,Rejeitada,Confirmada,4
classificacaoMunicipioResidencia,Rejeitada,Confirmada,Rejeitada,Confirmada,Rejeitada,Confirmada,3
RegiaoHospital,Confirmada,Confirmada,Rejeitada,Confirmada,Rejeitada,Rejeitada,3
INSTRUC,Rejeitada,Confirmada,Rejeitada,Confirmada,Confirmada,Confirmada,2
BASDIAGSP,Confirmada,Rejeitada,Confirmada,Confirmada,Rejeitada,Confirmada,2
TABAGISM,Rejeitada,Confirmada,Confirmada,Confirmada,Confirmada,Confirmada,1
Regiao,Confirmada,Confirmada,Rejeitada,Confirmada,Confirmada,Confirmada,1
SEXO,Confirmada,Confirmada,Confirmada,Confirmada,Confirmada,Rejeitada,1


In [ ]:
usarSelecaoNacional = False # True: aplica a seleção do Boruta Nacional em todas as regiões
 
for regiao, dados in dadosVarBin.items():
    resultadoBoruta = (dadosVarBin["Nacional"] if usarSelecaoNacional else dados)["resultadoBoruta"]
    selecionadas = resultadoBoruta.loc[resultadoBoruta["decisao"] != "Rejeitada", "variavel"] # Tentativas ficam: falta evidência para excluir
    rejeitadas = resultadoBoruta.loc[resultadoBoruta["decisao"] == "Rejeitada", "variavel"].tolist()
 
    X = dados["X"]
    categoricas = dados["categoricas"]
    numericas = dados["numericas"]
 
    dados["X"] = X.loc[:, X.columns.isin(selecionadas)]
    dados["categoricas"] = categoricas[categoricas.isin(selecionadas)]
    dados["numericas"] = numericas[numericas.isin(selecionadas)]
    dados["variaveisRejeitadas"] = rejeitadas
    print(f"{regiao} — {len(rejeitadas)} variáveis excluídas: {rejeitadas}")


In [ ]:
for regiao, dados in dadosVarBin.items():
    X = dados["X"]
    y = dados["y"]
    categoricas = dados["categoricas"]
    numericas = dados["numericas"]

    pipeCategoricas = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore')) # Transforma as colunas em números para a RF conseguir interpretar. handle unknown ignora colunas desconhecidas
    ])

    pipeNumericas = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)) # Preenche NaN pela mediana antes de entrar na RF
    ])

    preprocessador = ColumnTransformer([
        ('cat', pipeCategoricas, categoricas),
        ('num', pipeNumericas, numericas)
    ])

    arvoreEstocastica = Pipeline([
        ('preprocessador', preprocessador),
        ('rf', RandomForestClassifier(
            n_jobs=-1,
            random_state=rs
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=rs, stratify=y
    )

    dados["arvoreEstocastica"] = arvoreEstocastica
    dados["X_train"] = X_train
    dados["X_test"] = X_test
    dados["y_train"] = y_train
    dados["y_test"] = y_test

In [ ]:
for regiao, dados in dadosVarBin.items():
    X_train = dados["X_train"]
    y_train = dados["y_train"]
    X_test = dados["X_test"]
    y_test = dados["y_test"]
    arvoreEstocastica = dados["arvoreEstocastica"]

    arvoreEstocastica.fit(X_train, y_train) # Treinando os modelos
    pred = arvoreEstocastica.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, pred)
    print(f"{regiao} — AUC: {auc}")

    dados["auc"] = auc

In [ ]:
from joblib import parallel_backend

for regiao, dados in dadosVarBin.items():
    if "resultadoPermutacao" in dados:
        continue

    X_test = dados["X_test"]
    y_test = dados["y_test"]
    arvoreEstocastica = dados["arvoreEstocastica"]

    amostra = X_test.sample(10000, random_state=rs).index
    with parallel_backend("threading"):
        resultadoPermutacao = permutation_importance( # Importância por permutação
            arvoreEstocastica,
            X_test.loc[amostra],
            y_test.loc[amostra],
            n_repeats=3,
            random_state=rs,
            n_jobs=-1
        )
    dados["resultadoPermutacao"] = resultadoPermutacao
    print(f"{regiao} — permutação concluída")

In [ ]:
for regiao, dados in dadosVarBin.items():
    X_test = dados["X_test"]
    auc = dados["auc"]
    arvoreEstocastica = dados["arvoreEstocastica"]
    resultadoPermutacao = dados["resultadoPermutacao"]
    
    nomeVariavel = (  # Nomes das colunas depois do OneHotEncoder
        arvoreEstocastica.named_steps['preprocessador']
        .get_feature_names_out()
    )

    featureImportance = pd.DataFrame({ # Cria o DataFrame com os nomes e valores de importância associados
        "variavel": X_test.columns,
        "importancia": resultadoPermutacao.importances_mean
    })
    
    featureImportance["nome"] = ( # Limpa os nomes das variáveis
        featureImportance["variavel"]
        .str.replace("cat__", "", regex=False)
        .str.replace("num__", "", regex=False)
    )

    fiTopo = featureImportance.sort_values(by="importancia", ascending=False).head(20)
    fiTopo = fiTopo.sort_values(by="importancia", ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(fiTopo["nome"], fiTopo["importancia"], color='skyblue')
    plt.xlabel("Aumento no Erro (Permutation Importance)")
    plt.title(f"20 Variáveis mais relevantes para o atraso no tratamento — {regiao}")
    plt.tight_layout()
    plt.show()

    # Guarda tudo dentro do próprio dicionário da região
    dados["modelo"] = arvoreEstocastica
    dados["auc"] = auc
    dados["featureImportance"] = featureImportance
    dados["fiTopo"] = fiTopo